In [59]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from typing import Type, Optional, List

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder, PolynomialFeatures
from sklearn.decomposition import PCA
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import cross_val_score, GridSearchCV, StratifiedKFold
from sklearn.linear_model import ElasticNet
from sklearn.metrics import make_scorer

In [60]:
#Create a custom loss function to use on the class object
def rmsle(y_true : np.ndarray, y_pred : np.ndarray):
    power = np.power ( np.log(y_pred + 1) - np.log(y_true + 1) , 2)
    return np.sqrt(1/len(y_true) * np.sqrt(np.sum(power)))

score_rmsle = make_scorer(rmsle, greater_is_better=False)

In [61]:
class DataFit:
    '''
    This is a class to help streamline the entire model-fitting process and make it in the cleanest way possible
    '''

    ## Initialize the class, the main attributes to consider are the DataFrame object, as well as lists of the continous, categorical inputs to consider, and the output to focus on
    def __init__(self, df : pd.DataFrame, continous_inputs : List[str], categorical_inputs : List[str], output : str):
        #Create a copy of the originally provided DataFrame and select only the variables we will focus on. Store this new DataFrame into the df attribute
        self.df = df.loc[:,continous_inputs + categorical_inputs + [output]].copy()

        #Convert the categorical inputs into a categorical data type
        self.df.loc[:,categorical_inputs] = self.df.loc[:,categorical_inputs].astype("category")

        #Store the lists of inputs by type, as well as the outputs
        self.continous = continous_inputs
        self.categorical = categorical_inputs
        self.output = output

        #Define the resampling scheme
        self.my_cv = StratifiedKFold(n_splits=5, random_state=101, shuffle=True)


    ###########
    ## Define a method that will prepare the preprocessing based on user-defined pipelines
    def DefinePreprocessing(self, num_transform : Pipeline, cat_transform : Pipeline, remainder='drop'):
        '''
        Method that stores Pipelines with basic preprocessing operations for inputs, outputs, and categorical variables
        ColumnTransformer objects will be used to standarize the input and output variables when fitting the models (to avoid potential data leakage)
        '''

        self.cat_preprocessing = cat_transform

        self.cont_preprocessing = num_transform

        self._ColumnTransformerRemainder = remainder
        

    ###########
    ## Define a method that will take care of the splitting for us
    def DefineSplit(self):
        '''
        Since StratifiedKfold doesn't automatically recognize the groups in the training data (specially once the OneHotEncoding is performed), we need to use the split() method to provide a cross-validation generator
        (https://scikit-learn.org/stable/glossary.html#term-CV-splitter)
        This method will take the identified categorical variable and use it to generate the stratified k-fold scheme
        '''

        # Select the categorical data to use
        X_cat = self.df.loc[:,self.categorical].to_numpy(dtype=str).ravel()

        # Build the splits and return the indices
        Split_Generator = self.my_cv.split(self.df, X_cat)

        return(Split_Generator)


    ###########
    ## Define a function for the first ENET
    def FirstEnet(self):
        #Basic Estimator → Elastic Net
        enet_to_fit = ElasticNet(fit_intercept = True, max_iter = 50000)

        #Param_grid → For the enet, we test the l1 ratio and alpha
        enet_grid = {'enet__l1_ratio' : np.linspace(0,1, num=5),
                     'enet__alpha' : np.exp( np.linspace(-6, 6, num=11))}

        #Score → RMSLE, but Sklearn has the MSLE available, will transform it later
        Score = score_rmsle
        # Score = 'neg_root_mean_squared_error'

        #Build the column transformer for preprocessing
        #Create a copy of the continous pipeline (Standarization and maybe PCA)
        _temp_continous_pipeline_steps = self.cont_preprocessing.steps.copy()

        #Append to it the Polynomial features and final Standarization)
        _temp_continous_pipeline_steps.append(('Polynomial_Interactions', PolynomialFeatures(degree=2)))
        _temp_continous_pipeline_steps.append(('Standarize_Again', StandardScaler()))

        #Build the new Pipeline to use
        _temp_continous_pipeline = Pipeline(steps = _temp_continous_pipeline_steps)

        #Build the Preprocessing ColumnTransformer
        Preprocessing = ColumnTransformer( transformers = [ ('Manipulate_Inputs' , _temp_continous_pipeline, self.continous),
                                                            ('cat', self.cat_preprocessing, self.categorical)],
                                           remainder = self._ColumnTransformerRemainder)
                                        #    remainder = StandardScaler())

        #Complete Estimator → preprocess first,  then fit the enet
        enet_cv_wflow = Pipeline(steps = [('preprocessing', Preprocessing),
                                           ('enet', enet_to_fit)])
        
        #Define the GridSearchCV object
        self._FirstEnet_grid = GridSearchCV(estimator = enet_cv_wflow,
                                        param_grid = enet_grid,
                                        scoring=Score,
                                        cv=self.DefineSplit())

        #Fit
        self.FirstEnet_results = self._FirstEnet_grid.fit(X=self.df.drop(self.output, axis=1).copy(),
                                                          y=self.df.loc[:,self.output].copy())


    ###########
    ## Define a function for the second ENET
    def SecondEnet(self):
        #Basic Estimator → Elastic Net
        enet_to_fit = ElasticNet(fit_intercept = True, max_iter = 50000)

        #Param_grid → For the enet, we test the l1 ratio and alpha
        enet_grid = {'enet__l1_ratio' : np.linspace(0,1, num=5),
                     'enet__alpha' : np.exp( np.linspace(-6, 6, num=11))}

        #Score → RMSLE, but Sklearn has the MSLE available, will transform it later
        Score = score_rmsle

        #Build the column transformer for preprocessing
        #Create a copy of the continous pipeline (Standarization and maybe PCA)
        _temp_continous_pipeline_steps = self.cont_preprocessing.steps.copy()

        #Append to it the Polynomial features and final Standarization)
        _temp_continous_pipeline_steps.append(('Standarize_Again', StandardScaler()))

        #Build the new Pipeline to use
        _temp_continous_pipeline = Pipeline(steps = _temp_continous_pipeline_steps)

        #Build the Preprocessing ColumnTransformer
        Preprocessing = ColumnTransformer( transformers = [ ('Manipulate_Inputs' , _temp_continous_pipeline, self.continous),
                                                            ('cat', self.cat_preprocessing, self.categorical)],
                                           remainder = self._ColumnTransformerRemainder)

        #Complete Estimator → preprocess first,  then fit the enet
        enet_cv_wflow = Pipeline(steps = [('preprocessing', Preprocessing),
                                           ('enet', enet_to_fit)])
        
        #Define the GridSearchCV object
        self._SecondEnet_grid = GridSearchCV(estimator = enet_cv_wflow,
                                        param_grid = enet_grid,
                                        scoring=Score,
                                        cv=self.DefineSplit())

        #Fit
        self.SecondEnet_results = self._SecondEnet_grid.fit(X=self.df.drop(self.output, axis=1).copy(),
                                                          y=self.df.loc[:,self.output].copy())

    
    ###########
    ## Define a function for the second ENET
    def NN(self):
        #Basic Estimator → Elastic Net
        enet_to_fit = ElasticNet(fit_intercept = True, max_iter = 50000)

        #Param_grid → For the enet, we test the l1 ratio and alpha
        enet_grid = {'enet__l1_ratio' : np.linspace(0,1, num=5),
                     'enet__alpha' : np.exp( np.linspace(-6, 6, num=11))}

        #Score → RMSLE, but Sklearn has the MSLE available, will transform it later
        Score = score_rmsle

        #Build the column transformer for preprocessing
        #Create a copy of the continous pipeline (Standarization and maybe PCA)
        _temp_continous_pipeline_steps = self.cont_preprocessing.steps.copy()

        #Append to it the Polynomial features and final Standarization)
        _temp_continous_pipeline_steps.append(('Standarize_Again', StandardScaler()))

        #Build the new Pipeline to use
        _temp_continous_pipeline = Pipeline(steps = _temp_continous_pipeline_steps)

        #Build the Preprocessing ColumnTransformer
        Preprocessing = ColumnTransformer( transformers = [ ('Manipulate_Inputs' , _temp_continous_pipeline, self.continous),
                                                            ('cat', self.cat_preprocessing, self.categorical)],
                                           remainder = self._ColumnTransformerRemainder)

        #Complete Estimator → preprocess first,  then fit the enet
        enet_cv_wflow = Pipeline(steps = [('preprocessing', Preprocessing),
                                           ('enet', enet_to_fit)])
        
        #Define the GridSearchCV object
        self._SecondEnet_grid = GridSearchCV(estimator = enet_cv_wflow,
                                        param_grid = enet_grid,
                                        scoring=Score,
                                        cv=self.DefineSplit())

        #Fit
        self.SecondEnet_results = self._SecondEnet_grid.fit(X=self.df.drop(self.output, axis=1).copy(),
                                                          y=self.df.loc[:,self.output].copy())

***
## TEST

### Initialize the object

In [62]:
df_train = pd.read_csv('../train.csv')
df_train.head()

,id,spacegroup,number_of_total_atoms,percent_atom_al,percent_atom_ga,percent_atom_in,lattice_vector_1_ang,lattice_vector_2_ang,lattice_vector_3_ang,lattice_angle_alpha_degree,lattice_angle_beta_degree,lattice_angle_gamma_degree,formation_energy_ev_natom,bandgap_energy_ev
0,1,33,80.0,0.6250,0.3750,0.000,9.9523,8.5513,9.1775,90.0026,90.0023,90.0017,0.0680,3.4387
1,2,194,80.0,0.6250,0.3750,0.000,6.1840,6.1838,23.6287,90.0186,89.9980,120.0025,0.2490,2.9210
2,3,227,40.0,0.8125,0.1875,0.000,9.7510,5.6595,13.9630,90.9688,91.1228,30.5185,0.1821,2.7438
3,4,167,30.0,0.7500,0.0000,0.250,5.0036,5.0034,13.5318,89.9888,90.0119,120.0017,0.2172,3.3492
4,5,194,80.0,0.0000,0.6250,0.375,6.6614,6.6612,24.5813,89.9960,90.0006,119.9893,0.0505,1.3793


List of variables

In [63]:
output_list = ['formation_energy_ev_natom', 'bandgap_energy_ev']
categorical_list = ['spacegroup']
input_list = [var for var in df_train.columns.to_list() if var not in output_list+categorical_list+['id','number_of_total_atoms']]

Create an object

In [64]:
DF = DataFit(df_train, continous_inputs=input_list, categorical_inputs=categorical_list, output='bandgap_energy_ev')

### Preprocessing

In [65]:
# Define the pipeline
num_transform = Pipeline(steps = [ ('std_input', StandardScaler()) ])
cat_transform = Pipeline(steps = [ ('dummy', OneHotEncoder(drop='first')) ])

In [66]:
DF.DefinePreprocessing(num_transform=num_transform, cat_transform=cat_transform, remainder='drop')

### Split generator

In [67]:
DF.DefineSplit()

<generator object _BaseKFold.split at 0x000001E5293D9DD0>

In [68]:
DF.SecondEnet()

c:\Users\joe48\Anaconda3\envs\cmpinf\lib\site-packages\sklearn\linear_model\_coordinate_descent.py:648: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 1.109e+02, tolerance: 1.925e-01 Linear regression models with null weight for the l1 regularization term are more efficiently fitted using one of the solvers implemented in sklearn.linear_model.Ridge/RidgeCV instead.
  model = cd_fast.enet_coordinate_descent(
c:\Users\joe48\Anaconda3\envs\cmpinf\lib\site-packages\sklearn\linear_model\_coordinate_descent.py:648: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 1.097e+02, tolerance: 1.952e-01 Linear regression models with null weight for the l1 regularization term are more efficiently fitted using one of the solvers impl

In [69]:
pd.Series(DF.SecondEnet_results.predict(DF.df.drop(DF.output, axis=1).copy())).head()

0    3.131155
1    2.669748
2    2.679119
3    3.197939
4    0.969981
dtype: float64

In [70]:
DF.SecondEnet_results.best_score_

-0.07999581682644728

In [71]:
rmsle(DF.df[DF.output],
      DF.SecondEnet_results.predict(DF.df.drop(DF.output, axis=1).copy()))

0.053403570381835755